# Retraining Retention Current Audit v1

Purpose: align the current model-retention gate, SOXLBacktest qlib reference, dashboard runtime status, and next retraining handoff before launching more training.

This notebook is a current-state audit, not a promotion artifact. It proves whether any existing model can be retained and records the next training target under the strict requirements: complete `wf_2020` / `wf_2022` / `wf_2023_2024`, walk-forward MaxDD `< 20%`, YTD return above the SOXLBacktest qlib reference, and YTD return at least `95%` of the current best common-window model.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd())
CARD = ROOT / "runtime/model_selection/card/retraining_retention_current_audit_v1"
CARD.mkdir(parents=True, exist_ok=True)

GATE_SUMMARY_PATH = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
GATE_CSV_PATH = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/retention_gate.csv"
FRONTIER_SUMMARY_PATH = ROOT / "runtime/model_selection/card/model_retention_frontier_v1/summary.json"
FRONTIER_CSV_PATH = ROOT / "runtime/model_selection/card/model_retention_frontier_v1/retention_frontier.csv"
DASHBOARD_MANIFEST_PATH = ROOT / "dashboard_data/v2/manifest.json"
REPLAY_FRESHNESS_PATH = ROOT / "dashboard_data/model_replay_freshness.json"
LEGACY_METRICS_PATH = ROOT / "runtime/production/card/soxl_backtest_policy_reimplementation_v1/legacy_comparison_metrics.csv"
LEGACY_DAILY_PATH = ROOT / "runtime/production/card/soxl_backtest_policy_reimplementation_v1/new_policy_daily.csv"
PRIOR_HANDOFF_PATH = ROOT / "runtime/model_selection/card/retraining_gate_gap_handoff_v1/summary.json"

for path in [GATE_SUMMARY_PATH, GATE_CSV_PATH, FRONTIER_SUMMARY_PATH, FRONTIER_CSV_PATH, DASHBOARD_MANIFEST_PATH, REPLAY_FRESHNESS_PATH, LEGACY_METRICS_PATH, LEGACY_DAILY_PATH]:
    assert path.exists(), path

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n", encoding="utf-8")

gate_payload = read_json(GATE_SUMMARY_PATH)
gate_summary = gate_payload.get("summary", gate_payload)
frontier_payload = read_json(FRONTIER_SUMMARY_PATH)
frontier_summary = frontier_payload.get("summary", frontier_payload)
manifest = read_json(DASHBOARD_MANIFEST_PATH)
freshness = read_json(REPLAY_FRESHNESS_PATH)
prior_handoff = read_json(PRIOR_HANDOFF_PATH) if PRIOR_HANDOFF_PATH.exists() else {}

gate = pd.read_csv(GATE_CSV_PATH)
frontier = pd.read_csv(FRONTIER_CSV_PATH)
legacy_metrics = pd.read_csv(LEGACY_METRICS_PATH)
legacy_daily = pd.read_csv(LEGACY_DAILY_PATH)

print({
    "gate_profiles_scanned": gate_summary.get("profiles_scanned"),
    "gate_passing_profile_count": gate_summary.get("passing_profile_count"),
    "frontier_next_source": frontier_summary.get("recommended_next_source"),
    "dashboard_runnable": manifest.get("profile_groups", {}).get("runnable", []),
    "freshness_counts": freshness.get("counts"),
    "legacy_daily_rows": len(legacy_daily),
})

{'gate_profiles_scanned': 13, 'gate_passing_profile_count': 0, 'frontier_next_source': 'release16_risk_gated_state_head_objective_screen_v1', 'dashboard_runnable': [], 'freshness_counts': {'current': 0, 'refreshed': 0, 'failed': 0, 'skipped': 0}, 'legacy_daily_rows': 116}


## Current Retention Gate

The strict retention gate is authoritative for deciding whether any model should keep an executable actor. This cell classifies the current rows by missing walk-forward evidence, drawdown gap, SOXLBacktest return gap, and current-best alignment gap.

In [2]:
required_best = float(gate_summary["required_best_return_pct"])
reference_return = float(gate_summary["reference_return_pct"])
max_wf_dd = float(gate_summary["max_walkforward_drawdown_pct"])
current_best = float(gate_summary["current_best_return_pct"])

gate_current = gate.copy()
for col in ["ytd_return_pct", "ytd_max_drawdown_pct", "worst_wf_max_drawdown_pct", "missing_wf_window_count", "wf_drawdown_gap_pct", "best_return_gap_pct", "soxlbacktest_return_gap_pct"]:
    if col in gate_current:
        gate_current[col] = pd.to_numeric(gate_current[col], errors="coerce")

gate_current["has_complete_wf"] = gate_current["missing_wf_window_count"].fillna(0).eq(0)
gate_current["wf_dd_pass"] = gate_current["worst_wf_max_drawdown_pct"].lt(max_wf_dd)
gate_current["soxlbacktest_return_pass"] = gate_current["ytd_return_pct"].ge(reference_return)
gate_current["best_alignment_pass"] = gate_current["ytd_return_pct"].ge(required_best)
gate_current["strict_actor_retention_pass"] = gate_current["gate_pass"].astype(str).str.lower().eq("true")
gate_current["gap_priority"] = (
    gate_current["missing_wf_window_count"].fillna(0) * 1000
    + gate_current["wf_drawdown_gap_pct"].fillna(0) * 10
    + gate_current["best_return_gap_pct"].fillna(0)
    + gate_current["soxlbacktest_return_gap_pct"].fillna(0)
)

columns = [
    "profile", "status", "ytd_return_pct", "ytd_max_drawdown_pct", "wf_windows_present",
    "worst_wf_max_drawdown_pct", "missing_wf_window_count", "wf_drawdown_gap_pct",
    "best_return_gap_pct", "soxlbacktest_return_gap_pct", "has_complete_wf", "wf_dd_pass",
    "soxlbacktest_return_pass", "best_alignment_pass", "strict_actor_retention_pass", "retention_decision", "failures", "gap_priority",
]
gate_current_out = gate_current[[col for col in columns if col in gate_current.columns]].sort_values(
    ["strict_actor_retention_pass", "gap_priority", "ytd_return_pct"], ascending=[False, True, False]
)
gate_current_out.to_csv(CARD / "current_retention_gate.csv", index=False)
gate_current_out.head(14)

,profile,status,ytd_return_pct,ytd_max_drawdown_pct,wf_windows_present,worst_wf_max_drawdown_pct,missing_wf_window_count,wf_drawdown_gap_pct,best_return_gap_pct,soxlbacktest_return_gap_pct,has_complete_wf,wf_dd_pass,soxlbacktest_return_pass,best_alignment_pass,strict_actor_retention_pass,retention_decision,failures,gap_priority
0,soxl_backtest_policy_reimplementation_v1,reference,132.077978,13.554089,NaN,NaN,0,NaN,NaN,NaN,True,False,True,False,True,keep_reference_baseline,NaN,0.000000
4,mode_conditioned_rebound_reward_train_v2,ablation,177.573569,27.278641,"wf_2020,wf_2022,wf_2023_2024",21.831720,0,1.831720,35.569893,0.000000,True,False,True,False,False,retain_as_evidence_only,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...,53.887093
7,drawdown_rebound_reward_train_v1,ablation,172.832548,27.310377,"wf_2020,wf_2022,wf_2023_2024",23.546830,0,3.546830,40.310914,0.000000,True,False,True,False,False,retain_as_evidence_only,wf_max_drawdown_ge_20:wf_2020;ytd_return_below...,75.779214
3,state_conditioned_drawdown_rebound_reward_trai...,ablation,196.505054,31.390931,"wf_2020,wf_2022,wf_2023_2024",27.608986,0,7.608986,16.638408,0.000000,True,False,True,False,False,retain_as_evidence_only,wf_max_drawdown_ge_20:wf_2020;ytd_return_below...,92.728268
12,balanced_rule_residual_v22,canonical,116.882644,26.278174,"wf_2020,wf_2022,wf_2023_2024",54.665610,0,34.665610,96.260818,15.195334,True,False,False,False,False,retain_as_evidence_only,wf_max_drawdown_ge_20:wf_2020;wf_max_drawdown_...,458.112252
1,smoke,canonical,224.361539,30.346650,NaN,NaN,3,NaN,0.000000,0.000000,False,False,True,True,False,retain_as_evidence_only,missing_wf_window:wf_2020;missing_wf_window:wf...,3000.000000
2,balanced_macro_v1,candidate,196.749008,29.901822,NaN,NaN,3,NaN,16.394453,0.000000,False,False,True,False,False,retain_as_evidence_only,missing_wf_window:wf_2020;missing_wf_window:wf...,3016.394453
5,balanced_rule_residual_v19_reg_v22_retest_v1,ablation,175.012377,27.120494,NaN,NaN,3,NaN,38.131085,0.000000,False,False,True,False,False,retain_as_evidence_only,missing_wf_window:wf_2020;missing_wf_window:wf...,3038.131085
6,balanced_rule_residual_v19,archived,173.299227,24.399161,NaN,NaN,3,NaN,39.844235,0.000000,False,False,True,False,False,retain_as_evidence_only,missing_wf_window:wf_2020;missing_wf_window:wf...,3039.844235
8,balanced_rule_residual_v19_reg_light,candidate,164.253638,25.990693,NaN,NaN,3,NaN,48.889824,0.000000,False,False,True,False,False,retain_as_evidence_only,missing_wf_window:wf_2020;missing_wf_window:wf...,3048.889824


## SOXLBacktest qlib Reference Contract

The reference baseline must be treated as a deterministic buy/sell baseline, not a retained RL actor. The conversion is accepted only if the qlib daily replay matches the legacy windows on return, end position, trade count, and max drawdown.

In [3]:
delta_cols = ["return_delta_pct", "end_position_delta_pct", "trade_count_delta", "new_max_drawdown_pct", "legacy_max_drawdown_pct"]
legacy_check = legacy_metrics.copy()
for col in delta_cols:
    legacy_check[col] = pd.to_numeric(legacy_check[col], errors="coerce")
legacy_check["max_drawdown_delta_pct"] = legacy_check["new_max_drawdown_pct"] - legacy_check["legacy_max_drawdown_pct"]
legacy_check["qlib_reference_match"] = (
    legacy_check["return_delta_pct"].abs().le(1e-9)
    & legacy_check["end_position_delta_pct"].abs().le(1e-9)
    & legacy_check["trade_count_delta"].abs().le(0)
    & legacy_check["max_drawdown_delta_pct"].abs().le(1e-9)
)
legacy_check.to_csv(CARD / "soxlbacktest_qlib_alignment.csv", index=False)
legacy_alignment = {
    "windows": int(len(legacy_check)),
    "all_windows_match": bool(legacy_check["qlib_reference_match"].all()),
    "max_abs_return_delta_pct": float(legacy_check["return_delta_pct"].abs().max()),
    "max_abs_end_position_delta_pct": float(legacy_check["end_position_delta_pct"].abs().max()),
    "max_abs_max_drawdown_delta_pct": float(legacy_check["max_drawdown_delta_pct"].abs().max()),
    "daily_rows": int(len(legacy_daily)),
    "daily_start": str(legacy_daily.iloc[0]["date"] if "date" in legacy_daily.columns else legacy_daily.iloc[0].get("datetime")),
    "daily_end": str(legacy_daily.iloc[-1]["date"] if "date" in legacy_daily.columns else legacy_daily.iloc[-1].get("datetime")),
}
legacy_check

,window,legacy_window,start,end,new_return_pct,legacy_return_pct,return_delta_pct,new_end_position_pct,legacy_end_position_pct,end_position_delta_pct,new_trade_count,legacy_trade_count,trade_count_delta,new_max_drawdown_pct,legacy_max_drawdown_pct,max_drawdown_delta_pct,qlib_reference_match
0,ytd_2026,legacy_ytd_2026_fresh_start,2026-01-02,2026-06-09,132.077978,132.077978,0.000000e+00,15.91,15.91,0.0,3,3,0,13.554089,13.554089,0.000000e+00,True
1,june_2026,legacy_june_2026_fresh_start,2026-06-01,2026-06-09,3.138960,3.138960,4.440892e-15,29.94,29.94,0.0,1,1,0,1.528209,1.528209,-5.329071e-15,True


## Dashboard Runtime Contract

The dashboard can visualize evidence-only models, but currently no RL actor is runnable. The only selected-window rerun source is the deterministic SOXLBacktest reference baseline.

In [4]:
runtime_rows = pd.DataFrame(manifest.get("model_runtime_status", []))
if not runtime_rows.empty:
    runtime_rows.to_csv(CARD / "dashboard_model_runtime_status.csv", index=False)
else:
    pd.DataFrame().to_csv(CARD / "dashboard_model_runtime_status.csv", index=False)

profile_groups = manifest.get("profile_groups", {})
dashboard_runtime_contract = {
    "available_profile_count": len(manifest.get("available_profiles", [])),
    "runnable_profiles": profile_groups.get("runnable", []),
    "reference_profiles": profile_groups.get("reference", []),
    "evidence_only_profiles": profile_groups.get("evidence_only", []),
    "default_baseline_profile": manifest.get("default_baseline_profile"),
    "default_chart_profile": manifest.get("default_chart_profile"),
    "safe_backtest_end": manifest.get("safe_backtest_end"),
    "latest_market_date": manifest.get("latest_market_date"),
    "model_replay_freshness_counts": freshness.get("counts", {}),
    "model_replay_ready_count": sum(1 for row in freshness.get("profiles", []) if row.get("result") in {"current", "refreshed"}),
    "model_replay_profile_count": len(freshness.get("profiles", [])),
}
write_json(CARD / "dashboard_runtime_contract.json", dashboard_runtime_contract)
runtime_rows[[col for col in ["profile", "display_name", "role", "runnable", "runnable_reason", "source_kind", "feature_schema_signature", "row_count", "replay_start", "replay_end", "source_path"] if col in runtime_rows.columns]]

,profile,display_name,role,runnable,runnable_reason,source_kind,feature_schema_signature,row_count,replay_start,replay_end,source_path
0,soxl_backtest_policy_reimplementation_v1,SOXLBacktest Buy/Sell Baseline,reference,False,reference_baseline,soxlbacktest_qlib_buysell,0ede55bc8174,109,2026-01-02,2026-06-09,runtime/production/card/soxl_backtest_policy_r...
1,state_conditioned_drawdown_rebound_reward_trai...,state_conditioned_drawdown_rebound_reward_trai...,evidence_only,False,evidence_only_daily_replay,daily_replay_evidence,99a98f3b024c,1617,2020-01-02,2026-06-09,runtime/model/card/state_conditioned_drawdown_...
2,drawdown_rebound_reward_train_v1,drawdown_rebound_reward_train_v1,evidence_only,False,evidence_only_daily_replay,daily_replay_evidence,99a98f3b024c,1617,2020-01-02,2026-06-09,runtime/model/card/drawdown_rebound_reward_tra...
3,mode_conditioned_rebound_reward_train_v2,mode_conditioned_rebound_reward_train_v2,evidence_only,False,evidence_only_daily_replay,daily_replay_evidence,99a98f3b024c,1617,2020-01-02,2026-06-09,runtime/model/card/mode_conditioned_rebound_re...
4,balanced_rule_residual_v19_moe_v2,balanced_rule_residual_v19_moe_v2,evidence_only,False,evidence_only_retained_dashboard_export,retained_dashboard_export,99a98f3b024c,1603,2020-01-02,2026-05-19,dashboard_data/versioned/balanced_rule_residua...


## Frontier And Next Training Handoff

The frontier table decides whether the next action is more training, more diagnostics, or cleanup. The top current candidate is useful only as a diagnostic source: it has complete fold coverage and acceptable drawdown, but it fails relative-return gates in `wf_2020` and `wf_2023_2024`.

In [5]:
frontier_top = frontier.head(12).copy()
for col in ["worst_wf_max_drawdown_pct", "wf_drawdown_gap_pct", "min_wf_relative_return_pp", "relative_return_gap_to_target_pp", "ytd_return_pct", "ytd_gap_to_soxlbacktest_pct", "ytd_gap_to_best_pct"]:
    if col in frontier_top:
        frontier_top[col] = pd.to_numeric(frontier_top[col], errors="coerce")
frontier_top.to_csv(CARD / "frontier_top_candidates.csv", index=False)

recommended = frontier_top.iloc[0].to_dict() if len(frontier_top) else {}
training_handoff_current = {
    "decision": "diagnose_release16_failed_folds_before_full_retained_actor_training",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_training_allowed": False,
    "recommended_next_source": frontier_summary.get("recommended_next_source"),
    "recommended_next_action": frontier_summary.get("recommended_next_action"),
    "recommended_primary_blocker": frontier_summary.get("recommended_primary_blocker"),
    "recommended_failed_wf_windows": str(recommended.get("failed_wf_windows", "")).split(",") if recommended.get("failed_wf_windows") == recommended.get("failed_wf_windows") else [],
    "recommended_worst_wf_max_drawdown_pct": None if pd.isna(recommended.get("worst_wf_max_drawdown_pct")) else float(recommended.get("worst_wf_max_drawdown_pct")),
    "recommended_min_wf_relative_return_pp": None if pd.isna(recommended.get("min_wf_relative_return_pp")) else float(recommended.get("min_wf_relative_return_pp")),
    "recommended_relative_gap_to_target_pp": None if pd.isna(recommended.get("relative_return_gap_to_target_pp")) else float(recommended.get("relative_return_gap_to_target_pp")),
    "required_controls_before_training": [
        "Use official daily-path metrics as the selection basis, not raw backtest metrics.",
        "Diagnose wf_2020 and wf_2023_2024 relative-return gaps before launching a full retained actor run.",
        "Keep MaxDD < 20% across wf_2020/wf_2022/wf_2023_2024 as a hard gate.",
        "After any full actor run, rebuild model replays and dashboard bundle before retention review.",
        "Do not retain actor checkpoints until retention_gate.csv shows strict_actor_retention_pass=True.",
    ],
}
write_json(CARD / "training_handoff_current.json", training_handoff_current)
frontier_top[[col for col in ["frontier_rank", "experiment", "decision", "promotion_allowed", "primary_blocker", "next_action", "wf_windows_present", "failed_wf_windows", "worst_wf_max_drawdown_pct", "min_wf_relative_return_pp", "relative_return_gap_to_target_pp", "source_path"] if col in frontier_top.columns]]

,frontier_rank,experiment,decision,promotion_allowed,primary_blocker,next_action,wf_windows_present,failed_wf_windows,worst_wf_max_drawdown_pct,min_wf_relative_return_pp,relative_return_gap_to_target_pp,source_path
0,1,release16_risk_gated_state_head_objective_scre...,risk_gated_state_head_objective_screen_no_full...,False,walkforward_relative_return_gap,train_nonconvex_risk_gated_state_head_checkpoi...,"wf_2020,wf_2022,wf_2023_2024","wf_2020,wf_2023_2024",16.65,21.33,8.67,runtime/model_selection/card/release16_risk_ga...
1,2,cap10_wf2020_participation_failure_attribution_v1,wf2020_bull_participation_target_rebuild_requi...,False,missing_walkforward_evidence,rebuild_wf2020_bull_participation_target_befor...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/cap10_wf2020_part...
2,3,conservative_rebound_release_gate_v1,conservative_release_gate_candidate_no_training,False,missing_walkforward_evidence,revise_target_semantics_with_sparse_soxx_sprea...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/conservative_rebo...
3,4,constrained_checkpoint_selector_v1,constrained_checkpoint_selector_no_promotion,False,missing_walkforward_evidence,generate_reward_constrained_short_training_can...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/constrained_check...
4,5,fold_local_rebound_priority_conflict_v1,rebound_priority_conflict_blocks_bc_ppo,False,missing_walkforward_evidence,revise_state_objective_target_semantics_before...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/fold_local_reboun...
5,6,official_replay_bad_interval_feature_audit_v1,official_replay_bad_interval_state_feature_aud...,False,missing_walkforward_evidence,build official-replay-gated risk-state checkpo...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/official_replay_b...
6,7,official_replay_candidate_mechanism_pk_v1,near_miss_action_cap_overlay_is_best_diagnosti...,False,missing_walkforward_evidence,stress-test short-volshock action-cap overlay ...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/official_replay_c...
7,8,quantbench_ensemble_current_walkforward_v1,quantbench_current_walkforward_not_target_read...,False,missing_walkforward_evidence,use_failure_columns_to_tune_bull_release_or_dr...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/quantbench_ensemb...
8,9,release16_repaired_bc_warm_start_official_repl...,repaired_bc_warm_start_official_replay_failed_...,False,missing_walkforward_evidence,repair_replay_objective_with_drawdown_drift_an...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/release16_repaire...
9,10,short_volshock_action_cap_stress_test_v1,short_volshock_action_cap_near_miss_stress_fra...,False,missing_walkforward_evidence,reward-timing objective should target at least...,NaN,NaN,NaN,NaN,NaN,runtime/model_selection/card/short_volshock_ac...


## Requirement Status

This table is the compact audit against the objective. It intentionally treats indirect or partial evidence as incomplete.

In [6]:
requirements = [
    {
        "requirement": "SOXLBacktest converted to qlib reference baseline",
        "status": "pass" if legacy_alignment["all_windows_match"] else "fail",
        "evidence": "soxlbacktest_qlib_alignment.csv; legacy return/end-position/trade-count/MaxDD deltas are zero",
        "next_action": "Keep as reference baseline, not runnable RL actor",
    },
    {
        "requirement": "Only retain models with complete wf_2020/wf_2022/wf_2023_2024 evidence",
        "status": "fail",
        "evidence": f"passing_profile_count={gate_summary['passing_profile_count']} / profiles_scanned={gate_summary['profiles_scanned']}",
        "next_action": "No actor retention; evidence-only dashboard display only",
    },
    {
        "requirement": "Retained model walk-forward MaxDD < 20%",
        "status": "fail",
        "evidence": "No strict retained actor; closest static WF drawdown candidate still fails or lacks return alignment",
        "next_action": "Keep MaxDD <20 as hard gate; do not relax to rescue return",
    },
    {
        "requirement": "Retained model return aligns to current best and beats SOXLBacktest reference",
        "status": "fail",
        "evidence": f"required_best_return_pct={required_best:.6f}, reference_return_pct={reference_return:.6f}, passing_profiles=[]",
        "next_action": "Diagnose release16 wf_2020/wf_2023_2024 relative-return gaps before full training",
    },
    {
        "requirement": "Dashboard exposes current runnable/reference/evidence-only state",
        "status": "pass",
        "evidence": f"runnable={dashboard_runtime_contract['runnable_profiles']}, reference={dashboard_runtime_contract['reference_profiles']}, evidence_only_count={len(dashboard_runtime_contract['evidence_only_profiles'])}",
        "next_action": "Use Run source only for runnable/reference windows; chart evidence-only models separately",
    },
    {
        "requirement": "Next retraining target is evidence-backed, not a duplicate threshold tweak",
        "status": "partial",
        "evidence": f"frontier recommends {frontier_summary['recommended_next_source']} with blocker {frontier_summary['recommended_primary_blocker']}",
        "next_action": training_handoff_current["decision"],
    },
]
requirement_status = pd.DataFrame(requirements)
requirement_status.to_csv(CARD / "requirement_status.csv", index=False)
requirement_status

,requirement,status,evidence,next_action
0,SOXLBacktest converted to qlib reference baseline,pass,soxlbacktest_qlib_alignment.csv; legacy return...,"Keep as reference baseline, not runnable RL actor"
1,Only retain models with complete wf_2020/wf_20...,fail,passing_profile_count=0 / profiles_scanned=13,No actor retention; evidence-only dashboard di...
2,Retained model walk-forward MaxDD < 20%,fail,No strict retained actor; closest static WF dr...,Keep MaxDD <20 as hard gate; do not relax to r...
3,Retained model return aligns to current best a...,fail,"required_best_return_pct=213.143462, reference...",Diagnose release16 wf_2020/wf_2023_2024 relati...
4,Dashboard exposes current runnable/reference/e...,pass,"runnable=[], reference=['soxl_backtest_policy_...",Use Run source only for runnable/reference win...
5,"Next retraining target is evidence-backed, not...",partial,frontier recommends release16_risk_gated_state...,diagnose_release16_failed_folds_before_full_re...


## Persist Summary

In [7]:
summary = {
    "experiment": "retraining_retention_current_audit_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": training_handoff_current["decision"],
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "current_retention_gate": {
        "profiles_scanned": int(gate_summary["profiles_scanned"]),
        "passing_profile_count": int(gate_summary["passing_profile_count"]),
        "passing_profiles": gate_summary.get("passing_profiles", []),
        "reference_profile": gate_summary.get("reference_profile"),
        "reference_return_pct": reference_return,
        "current_best_return_pct": current_best,
        "required_best_return_pct": required_best,
        "max_walkforward_drawdown_pct": max_wf_dd,
        "required_wf_windows": gate_summary.get("required_wf_windows", []),
        "closest_wf_drawdown_candidate": gate_summary.get("closest_wf_drawdown_candidate"),
        "closest_ytd_return_candidate": gate_summary.get("closest_ytd_return_candidate"),
        "registry_keep_actor_count": gate_summary.get("registry_keep_actor_count"),
        "registry_keep_actor_strict_violation_count": gate_summary.get("registry_keep_actor_strict_violation_count"),
    },
    "soxlbacktest_qlib_reference": {
        **legacy_alignment,
        "role": "reference_baseline",
        "source_path": str(LEGACY_DAILY_PATH.relative_to(ROOT)),
        "metrics_path": str(LEGACY_METRICS_PATH.relative_to(ROOT)),
    },
    "dashboard_runtime_contract": dashboard_runtime_contract,
    "frontier": training_handoff_current,
    "prior_handoff_staleness": {
        "prior_path": str(PRIOR_HANDOFF_PATH.relative_to(ROOT)) if PRIOR_HANDOFF_PATH.exists() else None,
        "prior_profiles_scanned": prior_handoff.get("retention_gate", {}).get("profiles_scanned"),
        "current_profiles_scanned": gate_summary.get("profiles_scanned"),
        "prior_registry_keep_actor_count": prior_handoff.get("strict_retention_current_scope", {}).get("protected_keep_actor_exceptions"),
        "current_registry_keep_actor_count": gate_summary.get("registry_keep_actor_count"),
    },
    "sidecars": [
        "current_retention_gate.csv",
        "soxlbacktest_qlib_alignment.csv",
        "dashboard_model_runtime_status.csv",
        "dashboard_runtime_contract.json",
        "frontier_top_candidates.csv",
        "training_handoff_current.json",
        "requirement_status.csv",
    ],
}
write_json(CARD / "summary.json", summary)
print(json.dumps(summary, indent=2, ensure_ascii=False)[:4000])

{
  "experiment": "retraining_retention_current_audit_v1",
  "created_at": "2026-06-14T23:56:35.394376+00:00",
  "decision": "diagnose_release16_failed_folds_before_full_retained_actor_training",
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "current_retention_gate": {
    "profiles_scanned": 13,
    "passing_profile_count": 0,
    "passing_profiles": [],
    "reference_profile": "soxl_backtest_policy_reimplementation_v1",
    "reference_return_pct": 132.077978,
    "current_best_return_pct": 224.361539,
    "required_best_return_pct": 213.143462,
    "max_walkforward_drawdown_pct": 20.0,
    "required_wf_windows": [
      "wf_2020",
      "wf_2022",
      "wf_2023_2024"
    ],
    "closest_wf_drawdown_candidate": {
      "best_return_gap_pct": 35.569893,
      "missing_wf_window_count": 0,
      "profile": "mode_conditioned_rebound_reward_train_v2",
      "wf_drawdown_gap_pct": 1.83172,
      "worst_wf_max_drawdown_pct": 21.83172,
      "ytd_return_pct": 177.5735

## Conclusion

- **SOXLBacktest qlib conversion is accepted as a deterministic reference baseline**: `legacy_comparison_metrics.csv` shows zero return, end-position, trade-count, and MaxDD deltas for the checked YTD and June windows. It should remain a `reference_baseline`, not a runnable RL actor.
- **No existing model is retainable today**: current retention gate scans 13 profiles and has `passing_profile_count = 0`. Therefore no actor checkpoint should be kept as production/runnable on the basis of current evidence.
- **The explicit hard gate remains unchanged**: final retained models need complete `wf_2020`, `wf_2022`, `wf_2023_2024`, walk-forward MaxDD `< 20%`, YTD return above SOXLBacktest reference, and YTD return at least `95%` of the current best common-window model.
- **The next model-selection action is diagnostic, not full retained training**: the frontier points to `release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1`, but its selected fold-local candidate still fails relative-return gates in `wf_2020` and `wf_2023_2024`. The next run should diagnose those failed folds under official daily-path metrics before launching a full retained actor run.
- **Dashboard state is consistent with the gate**: there are no runnable RL profiles in the current manifest; SOXLBacktest is the reference rerun source, while RL exports are evidence-only comparison surfaces.